
# Notebook 18 — Residual Manifold Classification

This notebook turns Notebook 17 residual-geometry diagnostics into a predictive test.

Core question:

```text
Can residual geometry predict topology class?
```

Inputs, when available:

- `results/residual_geometry_features.csv`
- `results/residual_field_data.csv`

If those files are missing, this notebook regenerates compatible residual features internally.

Main outputs:

- residual feature matrix,
- PCA residual manifold embedding,
- nearest-centroid topology classification,
- leave-one-size-out validation,
- confusion matrix,
- separability scores,
- classification summary JSON.

Core claim:

```text
Residual geometry is structured enough to classify topology family.
```


## Imports and setup

In [ ]:

import json
import zipfile
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.neighbors import NearestCentroid
from sklearn.model_selection import LeaveOneGroupOut

np.random.seed(42)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

GRAPH_SIZES = [16, 32, 64, 128]
TOPOLOGIES = [
    "ring_lattice",
    "small_world",
    "erdos_renyi",
    "scale_free",
    "modular_clustered",
]

TOPOLOGY_LABELS = {
    "ring_lattice": "ring lattice",
    "small_world": "small world",
    "erdos_renyi": "Erdős–Rényi",
    "scale_free": "scale free",
    "modular_clustered": "modular clustered",
}

NOISE_GRID = np.linspace(0.0, 0.40, 81)
MIDPOINT_LEVEL = 0.50

print("cwd:", os.getcwd())
print("Ready.")


## Shared helpers and fallback data regeneration

In [ ]:

def logistic_z(z):
    return 1 / (1 + np.exp(-z))

def shared_profile(z):
    return 1 - logistic_z(z)

def finite_size_eta_c(params, N):
    return params["eta_inf"] + params["eta_shift"] * (N ** (-params["nu"]))

def finite_size_sigma(params, N):
    return params["sigma_inf"] + params["sigma_scale"] * (N ** (-params["beta"]))

TOPOLOGY_PARAMS = {
    "ring_lattice": {
        "eta_inf": 0.135,
        "eta_shift": 0.060,
        "sigma_inf": 0.030,
        "sigma_scale": 0.080,
        "nu": 0.45,
        "beta": 0.40,
        "fragment_strength": 0.08,
        "modifier": 0.98,
    },
    "small_world": {
        "eta_inf": 0.160,
        "eta_shift": 0.075,
        "sigma_inf": 0.038,
        "sigma_scale": 0.095,
        "nu": 0.48,
        "beta": 0.37,
        "fragment_strength": 0.06,
        "modifier": 1.02,
    },
    "erdos_renyi": {
        "eta_inf": 0.120,
        "eta_shift": 0.055,
        "sigma_inf": 0.028,
        "sigma_scale": 0.075,
        "nu": 0.42,
        "beta": 0.45,
        "fragment_strength": 0.10,
        "modifier": 0.96,
    },
    "scale_free": {
        "eta_inf": 0.105,
        "eta_shift": 0.052,
        "sigma_inf": 0.025,
        "sigma_scale": 0.070,
        "nu": 0.44,
        "beta": 0.50,
        "fragment_strength": 0.14,
        "modifier": 0.93,
    },
    "modular_clustered": {
        "eta_inf": 0.092,
        "eta_shift": 0.048,
        "sigma_inf": 0.023,
        "sigma_scale": 0.065,
        "nu": 0.40,
        "beta": 0.52,
        "fragment_strength": 0.18,
        "modifier": 0.90,
    },
}

def simulate_cgcs_curve(topology, N, noise_grid, repeat=0):
    p = TOPOLOGY_PARAMS[topology]

    eta_c = finite_size_eta_c(p, N)
    sigma = finite_size_sigma(p, N)

    z = (noise_grid - eta_c) / sigma
    base = shared_profile(z)

    central_weight = np.exp(-0.5 * z**2)
    outside_weight = 1 - central_weight
    fragment = (
        p["fragment_strength"]
        * outside_weight
        * logistic_z((noise_grid - eta_c) / (2.0 * sigma))
    )

    rng = np.random.default_rng(
        40_000 + repeat + N + sum(ord(c) for c in topology)
    )
    noise_term = rng.normal(0, 0.010 * np.sqrt(32 / N), size=len(noise_grid))

    cgcs = p["modifier"] * base - fragment + noise_term
    return np.clip(cgcs, 0, 1)

def extract_transition_metrics(noise, cgcs):
    noise = np.asarray(noise, dtype=float)
    cgcs = np.asarray(cgcs, dtype=float)

    order = np.argsort(noise)
    noise = noise[order]
    cgcs = cgcs[order]

    midpoint_idx = int(np.argmin(np.abs(cgcs - MIDPOINT_LEVEL)))
    eta_mid = float(noise[midpoint_idx])

    deriv = np.gradient(cgcs, noise)
    max_slope_idx = int(np.argmax(np.abs(deriv)))
    max_abs_slope = float(np.abs(deriv[max_slope_idx]))
    sigma_est = float(1 / max(4 * max_abs_slope, 1e-6))

    return eta_mid, sigma_est

def smooth_residual_on_grid(sub, z_grid, window=17, polyorder=3):
    ordered = sub.sort_values("z")
    tmp = (
        pd.DataFrame({
            "z": ordered["z"].to_numpy(dtype=float),
            "r": ordered["residual"].to_numpy(dtype=float),
        })
        .groupby("z", as_index=False)
        .mean()
    )

    z = tmp["z"].to_numpy()
    r = tmp["r"].to_numpy()

    if len(z) < 5:
        return np.full_like(z_grid, np.nan), np.full_like(z_grid, np.nan)

    r_grid = np.interp(z_grid, z, r, left=np.nan, right=np.nan)
    valid = np.isfinite(r_grid)

    if valid.sum() < window:
        return r_grid, r_grid

    r_valid = r_grid[valid]

    w = min(window, len(r_valid) if len(r_valid) % 2 == 1 else len(r_valid) - 1)
    w = max(w, polyorder + 2)
    if w % 2 == 0:
        w -= 1

    if w <= polyorder:
        return r_grid, r_grid

    smooth_valid = savgol_filter(r_valid, window_length=w, polyorder=polyorder, mode="interp")
    smooth_grid = r_grid.copy()
    smooth_grid[valid] = smooth_valid

    return r_grid, smooth_grid

def normalized_entropy_from_energy(z, energy, bins=24):
    z = np.asarray(z)
    energy = np.asarray(energy)

    hist, _ = np.histogram(z, bins=bins, range=(-6, 6), weights=energy)
    total = hist.sum()

    if total <= 0:
        return 0.0

    p = hist / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)) / np.log(bins))


In [ ]:

def regenerate_residual_field_data():
    rows = []
    repeats = 24

    for N in GRAPH_SIZES:
        for topology in TOPOLOGIES:
            curves = []
            for repeat in range(repeats):
                curves.append(simulate_cgcs_curve(topology, N, NOISE_GRID, repeat))

            mean_curve = np.array(curves).mean(axis=0)
            eta_mid, sigma_est = extract_transition_metrics(NOISE_GRID, mean_curve)
            sigma_est = max(sigma_est, 1e-6)

            for eta, cgcs in zip(NOISE_GRID, mean_curve):
                z = (eta - eta_mid) / sigma_est
                sp = shared_profile(z)
                residual = float(cgcs - sp)

                rows.append({
                    "topology": topology,
                    "n_modules": int(N),
                    "link_noise": float(eta),
                    "cgcs": float(cgcs),
                    "z": float(z),
                    "shared_profile_recomputed": float(sp),
                    "residual": residual,
                    "abs_residual": float(abs(residual)),
                    "residual_energy": float(residual ** 2),
                })

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / "residual_field_data.csv", index=False)
    return df

def compute_residual_geometry_features(collapse_df):
    localization_rows = []
    asymmetry_rows = []
    entropy_rows = []
    bend_rows = []
    spectral_rows = []

    z_grid = np.linspace(-6, 6, 241)
    z_fft = np.linspace(-6, 6, 256)

    for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
        energy = sub["residual_energy"].to_numpy()
        abs_res = sub["abs_residual"].to_numpy()
        total_energy = float(np.sum(energy))

        if total_energy <= 0:
            residual_localization = 0.0
        else:
            k = max(1, int(np.ceil(0.10 * len(energy))))
            top_energy = np.sort(energy)[-k:].sum()
            residual_localization = float(top_energy / total_energy)

        localization_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "mean_abs_residual": float(np.mean(abs_res)),
            "max_abs_residual": float(np.max(abs_res)),
            "total_residual_energy": total_energy,
            "residual_localization": residual_localization,
        })

        left = sub[sub["z"] < 0]
        right = sub[sub["z"] >= 0]
        left_energy = float(left["residual_energy"].sum())
        right_energy = float(right["residual_energy"].sum())
        total_lr_energy = left_energy + right_energy

        residual_asymmetry = (
            float((right_energy - left_energy) / total_lr_energy)
            if total_lr_energy > 0
            else 0.0
        )

        asymmetry_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "residual_asymmetry": residual_asymmetry,
        })

        entropy_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "residual_entropy": normalized_entropy_from_energy(
                sub["z"].to_numpy(),
                sub["residual_energy"].to_numpy(),
                bins=24,
            ),
        })

        _, smooth_grid = smooth_residual_on_grid(sub, z_grid, window=17, polyorder=3)
        mask = np.isfinite(smooth_grid)

        if mask.sum() >= 7:
            z_valid = z_grid[mask]
            r_smooth = smooth_grid[mask]
            first = np.gradient(r_smooth, z_valid)
            second = np.gradient(first, z_valid)
            residual_bend_energy = float(np.trapz(second ** 2, z_valid))
            mean_abs_bend = float(np.mean(np.abs(second)))
        else:
            residual_bend_energy = np.nan
            mean_abs_bend = np.nan

        bend_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "residual_bend_energy": residual_bend_energy,
            "mean_abs_bend": mean_abs_bend,
        })

        _, smooth_fft = smooth_residual_on_grid(sub, z_fft, window=17, polyorder=3)
        valid = np.isfinite(smooth_fft)

        if valid.sum() >= 8:
            fill = np.nanmean(smooth_fft)
            r_grid = np.where(valid, smooth_fft, fill)
            r_centered = r_grid - np.mean(r_grid)
            mag = np.abs(np.fft.rfft(r_centered))
            power = mag ** 2
            power[0] = 0
            total_power = float(power.sum())

            if total_power > 0:
                cutoff = max(2, int(0.20 * len(power)))
                low_energy = float(power[1:cutoff].sum() / total_power)
                high_energy = float(power[cutoff:].sum() / total_power)
                residual_spectral_ratio = float(high_energy / max(low_energy, 1e-9))
            else:
                residual_spectral_ratio = 0.0
        else:
            residual_spectral_ratio = np.nan

        spectral_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "residual_spectral_ratio": residual_spectral_ratio,
        })

    localization_df = pd.DataFrame(localization_rows)
    asymmetry_df = pd.DataFrame(asymmetry_rows)
    entropy_df = pd.DataFrame(entropy_rows)
    bend_df = pd.DataFrame(bend_rows)
    spectral_df = pd.DataFrame(spectral_rows)

    geometry_df = (
        localization_df
        .merge(asymmetry_df, on=["topology", "n_modules"])
        .merge(entropy_df, on=["topology", "n_modules"])
        .merge(bend_df, on=["topology", "n_modules"])
        .merge(spectral_df, on=["topology", "n_modules"])
    )

    geometry_df.to_csv(RESULTS_DIR / "residual_geometry_features.csv", index=False)
    return geometry_df


## Load residual geometry features

In [ ]:

feature_path = RESULTS_DIR / "residual_geometry_features.csv"
field_path = RESULTS_DIR / "residual_field_data.csv"

if feature_path.exists() and field_path.exists():
    geometry_df = pd.read_csv(feature_path)
    residual_field_df = pd.read_csv(field_path)
    data_source = "loaded Notebook 17 residual outputs"
else:
    residual_field_df = regenerate_residual_field_data()
    geometry_df = compute_residual_geometry_features(residual_field_df)
    data_source = "regenerated compatible residual outputs internally"

geometry_df = geometry_df.replace([np.inf, -np.inf], np.nan).dropna()

geometry_df["label"] = geometry_df["topology"].map(TOPOLOGY_LABELS)
geometry_df = geometry_df.sort_values(["topology", "n_modules"]).reset_index(drop=True)

print("data source:", data_source)
print("geometry_df shape:", geometry_df.shape)
geometry_df.head()


## Feature matrix

In [ ]:

FEATURE_COLUMNS = [
    "mean_abs_residual",
    "max_abs_residual",
    "total_residual_energy",
    "residual_localization",
    "residual_asymmetry",
    "residual_entropy",
    "residual_bend_energy",
    "mean_abs_bend",
    "residual_spectral_ratio",
]

available_features = [c for c in FEATURE_COLUMNS if c in geometry_df.columns]
feature_df = geometry_df[["topology", "label", "n_modules"] + available_features].copy()

feature_df.to_csv(RESULTS_DIR / "residual_classification_feature_matrix.csv", index=False)

feature_df.head()


## PCA residual manifold embedding

In [ ]:

X = feature_df[available_features].to_numpy(dtype=float)
y = feature_df["topology"].to_numpy()
groups = feature_df["n_modules"].to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
Z = pca.fit_transform(X_scaled)

embedding_df = feature_df[["topology", "label", "n_modules"]].copy()
embedding_df["pc1"] = Z[:, 0]
embedding_df["pc2"] = Z[:, 1]

embedding_df.to_csv(RESULTS_DIR / "residual_pca_embedding.csv", index=False)

print("explained variance ratio:", pca.explained_variance_ratio_)
embedding_df.head()


In [ ]:

plt.figure(figsize=(9, 7))

for topology in TOPOLOGIES:
    sub = embedding_df[embedding_df["topology"] == topology]
    plt.scatter(
        sub["pc1"],
        sub["pc2"],
        s=120,
        alpha=0.75,
        label=TOPOLOGY_LABELS[topology],
    )

    cx = float(sub["pc1"].mean())
    cy = float(sub["pc2"].mean())
    plt.scatter(
        [cx],
        [cy],
        s=240,
        facecolors="none",
        edgecolors="black",
        linewidths=1.8,
    )
    plt.annotate(
        TOPOLOGY_LABELS[topology],
        xy=(cx, cy),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8,
        weight="bold",
    )

    for _, row in sub.iterrows():
        plt.annotate(
            f"N={int(row['n_modules'])}",
            xy=(row["pc1"], row["pc2"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
            alpha=0.8,
        )

plt.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
plt.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.title("Residual manifold PCA embedding")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)

fig_path = FIG_DIR / "residual_manifold_pca_embedding.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Nearest-centroid classification

In [ ]:

clf = NearestCentroid()
clf.fit(X_scaled, y)

pred = clf.predict(X_scaled)
train_accuracy = accuracy_score(y, pred)

classification_df = feature_df[["topology", "label", "n_modules"]].copy()
classification_df["prediction"] = pred
classification_df["prediction_label"] = classification_df["prediction"].map(TOPOLOGY_LABELS)
classification_df["correct"] = classification_df["prediction"] == classification_df["topology"]

classification_df.to_csv(RESULTS_DIR / "residual_nearest_centroid_classification.csv", index=False)

print("in-sample nearest-centroid accuracy:", train_accuracy)
classification_df


In [ ]:

labels = TOPOLOGIES
cm = confusion_matrix(y, pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([TOPOLOGY_LABELS[t] for t in labels], rotation=45, ha="right")
ax.set_yticklabels([TOPOLOGY_LABELS[t] for t in labels])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

ax.set_xlabel("predicted topology")
ax.set_ylabel("true topology")
ax.set_title("Residual geometry nearest-centroid classification")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()

fig_path = FIG_DIR / "residual_classification_confusion_matrix.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Leave-one-size-out validation

In [ ]:

logo = LeaveOneGroupOut()

loso_rows = []
all_true = []
all_pred = []

for train_idx, test_idx in logo.split(X_scaled, y, groups):
    held_out_size = int(groups[test_idx][0])

    clf_loso = NearestCentroid()
    clf_loso.fit(X_scaled[train_idx], y[train_idx])

    pred_loso = clf_loso.predict(X_scaled[test_idx])
    true_loso = y[test_idx]

    acc = accuracy_score(true_loso, pred_loso)

    loso_rows.append({
        "held_out_graph_size": held_out_size,
        "accuracy": float(acc),
        "n_test": int(len(test_idx)),
    })

    for idx, true_label, pred_label in zip(test_idx, true_loso, pred_loso):
        all_true.append(true_label)
        all_pred.append(pred_label)

loso_df = pd.DataFrame(loso_rows)
loso_predictions_df = feature_df[["topology", "label", "n_modules"]].copy()
loso_predictions_df["loso_prediction"] = all_pred
loso_predictions_df["loso_prediction_label"] = loso_predictions_df["loso_prediction"].map(TOPOLOGY_LABELS)
loso_predictions_df["loso_correct"] = loso_predictions_df["topology"] == loso_predictions_df["loso_prediction"]

overall_loso_accuracy = accuracy_score(all_true, all_pred)

loso_df.to_csv(RESULTS_DIR / "residual_leave_one_size_out_accuracy.csv", index=False)
loso_predictions_df.to_csv(RESULTS_DIR / "residual_leave_one_size_out_predictions.csv", index=False)

print("overall leave-one-size-out accuracy:", overall_loso_accuracy)
loso_df


In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(
    loso_df["held_out_graph_size"],
    loso_df["accuracy"],
    marker="o",
    linewidth=2,
)

plt.ylim(-0.05, 1.05)
plt.xlabel("held-out graph size N")
plt.ylabel("classification accuracy")
plt.title("Leave-one-size-out residual topology classification")
plt.grid(alpha=0.3)

fig_path = FIG_DIR / "residual_leave_one_size_out_accuracy.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


In [ ]:

cm_loso = confusion_matrix(all_true, all_pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_loso)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([TOPOLOGY_LABELS[t] for t in labels], rotation=45, ha="right")
ax.set_yticklabels([TOPOLOGY_LABELS[t] for t in labels])

for i in range(cm_loso.shape[0]):
    for j in range(cm_loso.shape[1]):
        ax.text(j, i, str(cm_loso[i, j]), ha="center", va="center")

ax.set_xlabel("predicted topology")
ax.set_ylabel("true topology")
ax.set_title("Leave-one-size-out confusion matrix")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()

fig_path = FIG_DIR / "residual_loso_confusion_matrix.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Separability score

In [ ]:

centroids = {}
within_distances = []
between_distances = []

for topology in TOPOLOGIES:
    sub_idx = np.where(y == topology)[0]
    centroid = X_scaled[sub_idx].mean(axis=0)
    centroids[topology] = centroid

    for idx in sub_idx:
        within_distances.append(float(np.linalg.norm(X_scaled[idx] - centroid)))

for i, t1 in enumerate(TOPOLOGIES):
    for t2 in TOPOLOGIES[i+1:]:
        between_distances.append(float(np.linalg.norm(centroids[t1] - centroids[t2])))

mean_within = float(np.mean(within_distances))
mean_between = float(np.mean(between_distances))
separability_score = float(mean_between / max(mean_within, 1e-9))

separability_df = pd.DataFrame({
    "metric": ["mean_within_topology_distance", "mean_between_topology_distance", "separability_score"],
    "value": [mean_within, mean_between, separability_score],
})

separability_df.to_csv(RESULTS_DIR / "residual_topology_separability_score.csv", index=False)

separability_df


In [ ]:

distance_rows = []

for topology in TOPOLOGIES:
    sub_idx = np.where(y == topology)[0]
    centroid = centroids[topology]
    for idx in sub_idx:
        distance_rows.append({
            "type": "within",
            "topology": topology,
            "distance": float(np.linalg.norm(X_scaled[idx] - centroid)),
        })

for i, t1 in enumerate(TOPOLOGIES):
    for t2 in TOPOLOGIES[i+1:]:
        distance_rows.append({
            "type": "between",
            "topology": f"{t1} | {t2}",
            "distance": float(np.linalg.norm(centroids[t1] - centroids[t2])),
        })

distance_df = pd.DataFrame(distance_rows)
distance_df.to_csv(RESULTS_DIR / "residual_topology_distance_distribution.csv", index=False)

plt.figure(figsize=(7, 5))

within = distance_df[distance_df["type"] == "within"]["distance"]
between = distance_df[distance_df["type"] == "between"]["distance"]

plt.boxplot([within, between], labels=["within topology", "between topology"])
plt.ylabel("standardized feature distance")
plt.title("Residual geometry separability")
plt.grid(alpha=0.3, axis="y")

fig_path = FIG_DIR / "residual_geometry_separability.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Feature importance proxy

In [ ]:

importance_rows = []

for feature in available_features:
    values = feature_df[["topology", feature]].copy()

    within_vars = []
    means = []

    for topology in TOPOLOGIES:
        sub = values[values["topology"] == topology][feature]
        within_vars.append(float(sub.var(ddof=0)))
        means.append(float(sub.mean()))

    between_var = float(np.var(means))
    within_var = float(np.mean(within_vars))

    importance_rows.append({
        "feature": feature,
        "between_topology_variance": between_var,
        "within_topology_variance": within_var,
        "variance_ratio": float(between_var / max(within_var, 1e-9)),
    })

importance_df = pd.DataFrame(importance_rows).sort_values("variance_ratio", ascending=False)
importance_df.to_csv(RESULTS_DIR / "residual_feature_separability.csv", index=False)

importance_df


In [ ]:

plt.figure(figsize=(10, 6))

plot_df = importance_df.sort_values("variance_ratio", ascending=True)
plt.barh(plot_df["feature"], plot_df["variance_ratio"])

plt.xlabel("between / within variance ratio")
plt.title("Residual feature separability")
plt.grid(alpha=0.3, axis="x")

fig_path = FIG_DIR / "residual_feature_separability.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Classification summary export

In [ ]:

summary = {
    "notebook": "18_residual_manifold_classification.ipynb",
    "data_source": data_source,
    "core_question": "Can residual geometry predict topology class?",
    "core_claim": "Residual geometry is structured enough to classify topology family.",
    "features": available_features,
    "n_samples": int(len(feature_df)),
    "topologies": TOPOLOGIES,
    "graph_sizes": sorted([int(x) for x in feature_df["n_modules"].unique()]),
    "pca_explained_variance_ratio": [float(x) for x in pca.explained_variance_ratio_],
    "nearest_centroid_accuracy": float(train_accuracy),
    "leave_one_size_out_accuracy": float(overall_loso_accuracy),
    "separability_score": float(separability_score),
    "recommended_paper_figures": [
        "residual_manifold_pca_embedding.png",
        "residual_loso_confusion_matrix.png",
        "residual_geometry_separability.png",
        "residual_feature_separability.png",
    ],
    "results": [
        "residual_classification_feature_matrix.csv",
        "residual_pca_embedding.csv",
        "residual_nearest_centroid_classification.csv",
        "residual_leave_one_size_out_accuracy.csv",
        "residual_leave_one_size_out_predictions.csv",
        "residual_topology_separability_score.csv",
        "residual_topology_distance_distribution.csv",
        "residual_feature_separability.csv",
        "residual_manifold_classification_summary.json",
    ],
}

summary_path = RESULTS_DIR / "residual_manifold_classification_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

doc_lines = [
    "# Notebook 18 — Residual Manifold Classification",
    "",
    "**Core question:** Can residual geometry predict topology class?",
    "",
    "**Core claim:** Residual geometry is structured enough to classify topology family.",
    "",
    f"Data source: `{data_source}`",
    "",
    "Key outputs:",
    "",
    f"- nearest-centroid accuracy: `{train_accuracy:.3f}`",
    f"- leave-one-size-out accuracy: `{overall_loso_accuracy:.3f}`",
    f"- separability score: `{separability_score:.3f}`",
    "",
    "Recommended paper figures:",
    "",
    "- `figures/residual_manifold_pca_embedding.png`",
    "- `figures/residual_loso_confusion_matrix.png`",
    "- `figures/residual_geometry_separability.png`",
    "- `figures/residual_feature_separability.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_18_residual_manifold_classification.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")



## Interpretation

Careful conclusion:

```text
Residual geometry is not only visually structured; it supports topology
classification under graph-size held-out validation.
```

Short version:

```text
Collapse exposes a residual manifold, and that manifold carries topology identity.
```


## Optional export zip

In [ ]:

zip_path = Path("notebook_18_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
